# 📉 Customer Churn Prediction
> **Goal:** Predict which customers are likely to leave a telecom company, and understand *why* — to enable proactive retention strategies.

**Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**Author:** Yasmine Pérez  
**Stack:** Python · Pandas · Scikit-learn · XGBoost · SHAP · Matplotlib · Seaborn

---
## 📋 Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Preprocessing & Feature Engineering](#3)
4. [Modeling: Logistic Regression, Random Forest, XGBoost](#4)
5. [Evaluation & Threshold Optimization](#5)
6. [Model Interpretability with SHAP](#6)
7. [Business Insights & Recommendations](#7)

---
<a id='1'></a>
## 1. Setup & Data Loading

In [ ]:
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score
)

# Interpretability
import shap

print('✅ Libraries loaded successfully')

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape: {df.shape}')
print(f'Churn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')
df.head()

---
<a id='2'></a>
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic overview
print('=== Data Types ===')
print(df.dtypes.value_counts())
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Target Distribution ===')
print(df['Churn'].value_counts())

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Churn'].value_counts()
colors = ['#4C9BE8', '#E84C4C']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution (counts)', fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Distribution (%)', fontweight='bold')

plt.suptitle('⚠️ Class Imbalance: ~73% No Churn vs ~27% Churn', fontsize=11, color='gray')
plt.tight_layout()
plt.savefig('img/churn_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Numerical features distribution by churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Fix TotalCharges (loaded as object)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    for label, color in zip(['No', 'Yes'], ['#4C9BE8', '#E84C4C']):
        subset = df[df['Churn'] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='none')
    ax.set_title(f'{col} by Churn', fontweight='bold')
    ax.legend(title='Churn')
    ax.set_xlabel(col)

plt.suptitle('Numerical Features Distribution by Churn', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('img/numerical_distributions.png', bbox_inches='tight')
plt.show()

print('\n💡 Key observations:')
print('  - Low-tenure customers churn more → early months are critical')
print('  - High monthly charges → higher churn risk')
print('  - Low total charges + high monthly → new customers paying a lot')

In [ ]:
# Categorical features: churn rate per category
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)
    bars = ax.barh(churn_rate.index, churn_rate.values * 100,
                   color=plt.cm.RdYlGn_r(churn_rate.values), edgecolor='none')
    ax.set_xlabel('Churn Rate (%)')
    ax.set_title(col, fontweight='bold')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, val in zip(bars, churn_rate.values):
        ax.text(val * 100 + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Churn Rate by Categorical Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('img/categorical_churn_rates.png', bbox_inches='tight')
plt.show()

---
<a id='3'></a>
## 3. Preprocessing & Feature Engineering

In [ ]:
# --- Clean & Encode ---
df_model = df.copy()

# Drop customerID
df_model.drop('customerID', axis=1, inplace=True)

# Fix TotalCharges
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model['TotalCharges'].fillna(df_model['TotalCharges'].median(), inplace=True)

# Target encode
df_model['Churn'] = (df_model['Churn'] == 'Yes').astype(int)

# Feature engineering
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)
df_model['IsNewCustomer'] = (df_model['tenure'] <= 6).astype(int)
df_model['HasMultipleServices'] = (
    (df_model['OnlineSecurity'] == 'Yes').astype(int) +
    (df_model['TechSupport'] == 'Yes').astype(int) +
    (df_model['StreamingTV'] == 'Yes').astype(int) +
    (df_model['StreamingMovies'] == 'Yes').astype(int)
)

# Separate features
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Categorical features ({len(cat_features)}): {cat_features}')
print(f'Numerical features ({len(num_features)}): {num_features}')
print(f'\nTarget balance: {y.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}')

---
<a id='4'></a>
## 4. Modeling: Logistic Regression, Random Forest, XGBoost

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'XGBoost':             XGBClassifier(scale_pos_weight=3, use_label_encoder=False,
                                         eval_metric='logloss', random_state=42, verbosity=0)
}

# Cross-validation with StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

for name, model in models.items():
    scores = cross_val_score(model, X_train_proc, y_train, cv=cv, scoring='roc_auc')
    results[name] = scores
    print(f'{name:25} | AUC-ROC: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Fit all models on full train set
fitted_models = {}
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    fitted_models[name] = model

# ROC curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4C9BE8', '#F5A623', '#E84C4C']
for (name, model), color in zip(fitted_models.items(), colors):
    y_proba = model.predict_proba(X_test_proc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves Comparison', fontweight='bold')
axes[0].legend()

# Precision-Recall curves
for (name, model), color in zip(fitted_models.items(), colors):
    y_proba = model.predict_proba(X_test_proc)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    axes[1].plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=color, lw=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/roc_pr_curves.png', bbox_inches='tight')
plt.show()

---
<a id='5'></a>
## 5. Evaluation & Threshold Optimization

> 💡 **Business context:** Missing a churner is more costly than a false alarm.  
> We optimize the threshold to **maximize F2-score** (weights Recall more than Precision).

In [ ]:
# Use XGBoost as final model (best AUC)
best_model = fitted_models['XGBoost']
y_proba = best_model.predict_proba(X_test_proc)[:, 1]

# Find threshold that maximizes F2
thresholds = np.arange(0.1, 0.9, 0.01)
f2_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f2 = f1_score(y_test, y_pred_t, beta=2, zero_division=0)
    f2_scores.append(f2)

best_threshold = thresholds[np.argmax(f2_scores)]
print(f'Optimal threshold (F2): {best_threshold:.2f}')

# Final predictions
y_pred_final = (y_proba >= best_threshold).astype(int)

# Threshold plot
plt.figure(figsize=(9, 4))
plt.plot(thresholds, f2_scores, color='#A78BFA', lw=2)
plt.axvline(best_threshold, color='#E84C4C', linestyle='--', label=f'Best threshold = {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F2 Score')
plt.title('Threshold Optimization (F2 Score)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('img/threshold_optimization.png', bbox_inches='tight')
plt.show()

In [ ]:
# Classification report & Confusion Matrix
print('=== Classification Report (XGBoost, optimized threshold) ===')
print(classification_report(y_test, y_pred_final, target_names=['No Churn', 'Churn']))

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'], ax=ax)
ax.set_title('Confusion Matrix — XGBoost (optimized threshold)', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('img/confusion_matrix.png', bbox_inches='tight')
plt.show()

---
<a id='6'></a>
## 6. Model Interpretability with SHAP

In [ ]:
# SHAP values
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_proc)

# Get feature names after one-hot encoding
ohe_features = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_features)
all_features = num_features + list(ohe_features)

# Summary plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_proc, feature_names=all_features,
                  max_display=15, show=False)
plt.title('SHAP Feature Importance — Top 15', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('img/shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# Individual prediction explanation (a churner example)
churner_idx = np.where(y_test.values == 1)[0][0]

print(f'Explaining prediction for customer index {churner_idx}')
print(f'Actual: Churn={y_test.values[churner_idx]} | Predicted prob: {y_proba[churner_idx]:.2%}')

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[churner_idx],
    X_test_proc[churner_idx],
    feature_names=all_features,
    matplotlib=True,
    show=False,
    figsize=(18, 4)
)
plt.title('Individual SHAP Explanation — Churner', fontweight='bold')
plt.tight_layout()
plt.savefig('img/shap_force_plot.png', bbox_inches='tight')
plt.show()

---
<a id='7'></a>
## 7. Business Insights & Recommendations

Based on EDA and SHAP analysis, the most impactful churn drivers are:

| Driver | Insight | Recommended Action |
|---|---|---|
| **Contract type** | Month-to-month customers churn 3x more | Offer discounts to switch to annual plans |
| **Tenure** | First 6 months are critical | Early onboarding program + check-in calls |
| **Monthly charges** | High charges without perceived value drive churn | Bundle discounts for high-spending customers |
| **Tech Support / Security** | Customers without add-ons churn more | Offer free trial of support services |
| **Payment method** | Electronic check → highest churn rate | Incentivize auto-pay methods |

### 💰 Estimated Business Impact

Assuming ~7,000 customers, 27% churn rate, and \$65 avg monthly revenue:
- Baseline churn cost: **\$122K/month**
- If model captures 70% of churners and 40% are retained with intervention:
- Estimated savings: **~\$34K/month** with targeted retention campaigns

In [ ]:
# Final model summary
print('=' * 55)
print('         FINAL MODEL SUMMARY — XGBoost')
print('=' * 55)
print(f'  AUC-ROC:          {roc_auc_score(y_test, y_proba):.4f}')
print(f'  Optimal threshold: {best_threshold:.2f}')
print(f'  Recall (Churn):   {(y_pred_final[y_test==1]==1).mean():.4f}')
print(f'  Precision (Churn):{(y_test[y_pred_final==1]==1).mean():.4f}')
print('=' * 55)
print('\n✅ Model ready for deployment or integration into a Streamlit dashboard.')